<img src="images/nvidia_header.png" style="margin-left: -30px; width: 300px; float: left;">

# CVAT Client for Videos 

This notebook will walk you through how to use CVAT client with MONAI Label.

<img src="./images/monailabel_arch_CVAT.png" width="1000"/>

---

> **Note**: This notebook is an adaptation of [monailabel_endoscopy_cvat_tooltracking.ipynb](../toolkit/tutorials/monai/monailabel/monailabel_endoscopy_cvat_tooltracking.ipynb) 
notebook released with MONAI Toolkit. 
We have removed cells for setup and data download which we have already down for you in this lab. 

---

### Table of Contents
**[1.0 CVAT Background](#1.0-CVAT-Background)** <br>
&nbsp;&nbsp;&nbsp;&nbsp;[1.1 MONAI Label Endoscopy App - Surgical Tool Segmentation with CVAT](#1.1-MONAI-Label-Endoscopy-App---Surgical-Tool-Segmentation-with-CVAT) <br>
**[2.0 Set Up CVAT and Nuclio](#2.0-Set-Up-CVAT-and-Nuclio)** <br>
&nbsp;&nbsp;&nbsp;&nbsp;[2.1 Set the Host and CVAT Version, Configure CVAT](#2.1-Set-the-Host-and-CVAT-Version,-Configure-CVAT) <br>
&nbsp;&nbsp;&nbsp;&nbsp;[2.2 Modify Docker Configuration](#2.2-Modify-Docker-Configuration) <br>
&nbsp;&nbsp;&nbsp;&nbsp;[2.3 Start Docker for CVAT](#2.3-Start-Docker-for-CVAT) <br>
&nbsp;&nbsp;&nbsp;&nbsp;[2.4 Create a CVAT User Using Terminal](#2.4-Create-a-CVAT-User-Using-Terminal) <br>
&nbsp;&nbsp;&nbsp;&nbsp;[2.5 Test CVAT](#2.5-Test-CVAT) <br>
&nbsp;&nbsp;&nbsp;&nbsp;[2.6 Test Nuclio](#2.6-Test-Nuclio) <br>
**[3.0 Set Up Nuclio Pipelines with MONAI Models](#3.0-Set-Up-Nuclio-Pipelines-with-MONAI-Models)** <br>
&nbsp;&nbsp;&nbsp;&nbsp;[3.1 Start MONAI Label Server](#3.1-Start-MONAI-Label-Server) <br>
&nbsp;&nbsp;&nbsp;&nbsp;[3.2 Set Up Nuclio Pipelines](#3.2-Set-Up-Nuclio-Pipelines) <br>
**[4.0 Active Learning Workflow with MONAI Label + CVAT](#4.0-Active-Learning-Workflow-with-MONAI-Label-+-CVAT)** <br>
&nbsp;&nbsp;&nbsp;&nbsp;[4.1 Start MONAI Label Server for Active Learning](#4.1-Start-MONAI-Label-Server-for-Active-Learning) <br>
&nbsp;&nbsp;&nbsp;&nbsp;[4.2 Labeling with CVAT and MONAI Label AI Tool](#4.2-Labeling-with-CVAT-and-MONAI-Label-AI-Tool) <br>
&nbsp;&nbsp;&nbsp;&nbsp;[4.3 Trigger Model Fine-Tuning](#4.3-Trigger-Model-Fine-Tuning) <br>
&nbsp;&nbsp;&nbsp;&nbsp;[4.4 Scoring and Create Next Iteration for CVAT](#4.4-Scoring-and-Create-Next-Iteration-for-CVAT) <br>
&nbsp;&nbsp;&nbsp;&nbsp;[4.5 Manually Upload Your Images and Create Task](#4.5-Manually-Upload-Your-Images-and-Create-Task) <br>
**[5.0 Shut Down CVAT](#5.0-Shut-Down-CVAT)** <br>
&nbsp;&nbsp;&nbsp;&nbsp;[5.1 Remove Containers](#5.1-Remove-Containers) <br>
&nbsp;&nbsp;&nbsp;&nbsp;[5.2 Remove Volumes](#5.2-Remove-Volumes) <br>

---
# 1.0 CVAT Background

## 1.1 MONAI Label Endoscopy App - Surgical Tool Segmentation with CVAT

***The computer-assisted intervention (CAI) workflow with MONAI Label***

In this notebook, we provide the endoscopy app example with surgical tool segmentation. The tooltracking bundle is used for the demonstration. CVAT + Nuclio is the client viewer and function deployment platform, while MONAI Label provides interactive labeling support for endoscopic applications.


**Surgical Tool Segmentation with the tooltracking bundle:**

This model is trained using a flexible UNet structure with a UNet architecture decoder. Datasets use private samples from Activ Surgical.

In this tutorial, we provide publicly available sample data for demonstration. We highlight the active learning and continuous learning process with MONAI Label + Endoscopic integration.

![cvat](../toolkit/tutorials/monailabel/figures/monailabel_endoscopy_cvat/cvat.png)

---
# 2.0 Set Up CVAT and Nuclio

## 2.1 Set the Host and CVAT Version, Configure CVAT

To start the CVAT Container we have exposed the docker socket and installed the docker CLI in this cloud instance. 
This allows you to run docker commands from JupyterLab and launch sibling containers. 

In [ ]:
import os
HOST_NAME=os.environ.get('HOST_NAME')
HOST_IP=os.getenv('HOST_IP')
# %env CVAT_HOST={HOST_NAME}
if len(HOST_NAME)==0:
    from requests import get
    HOST_NAME=get('https://api.ipify.org').text
    HOST_IP=HOST_NAME
HOST_NAME=HOST_IP ## cvat doesn't work with host name     
print(f" HOST_NAME is {HOST_NAME} and IP {HOST_IP}")

%env CVAT_HOST={HOST_IP}
# %env CVAT_VERSION=v2.1.0
CVAT_DOCKER_DIR="/cvat_on_host"

DLI_DATA_DIR=os.environ.get('DLI_DATA_DIR')

## 2.2 Modify Docker Configuration
You should skip this step if you are running on your machine or you have access to the host terminal.
However, since we are running in a DLI cloud instance, and already in a container,  
we mounted the docker socket and can call docker commands using the docker daemon on the host.
We need to use the host path and not the path you see in this JupyterLab container.    

To simplify, we will just copy the CVAT cloned repo to different path 
We have created a path `/cvat_on_host` which exists on the host as well as in the JupyterLab container we are in. 
We will use it to launch `docker-compose` correctly.

In [ ]:
!cp -r /dli/data/cvat/* $CVAT_DOCKER_DIR 

## 2.3 Start Docker for CVAT

In [ ]:
print(CVAT_DOCKER_DIR)
%cd $CVAT_DOCKER_DIR/
!ls -la docker*

In [ ]:
!docker compose -p cvat -f docker-compose.yml -f components/serverless/docker-compose.serverless.yml up -d

In [ ]:
!docker ps |grep cvat

## 2.4 Create a CVAT User Using Terminal
To create a user for CVAT, you need to open a terminal and run command below.
This would promt you to add a user, email and password
<div class="alert alert-block alert-info">
    
```
docker exec -it cvat_server bash -ic 'python3 ~/manage.py createsuperuser'
```

</div>
You can skip email and have a short username and passward as admin/admin.

## 2.5 Test CVAT 

In [ ]:
import ipywidgets
url_text='<h1><center><a href="http://'+HOST_NAME+':8080/" target="_blank">'
url_text=url_text+'<img src="https://neovision.fr/wp-content/uploads/2021/05/CVAT-Logo-complet-300x83.png" width="100" height="50"/>'
url_text=url_text+'<br>Click to open CVAT <br> in a new window</a></center></h1>'
ipywidgets.HTML(url_text)

You should see the page below were you can log in with your user/password. 

<img src="./images/cvat_login.png" width="600"/>

## 2.6 Test Nuclio
Nuclio is the tool which CVAT uses to serve models. We will add some models to it below. For now just check that you can access the site.

In [ ]:
url_text='<h1><center><a href="http://'+HOST_NAME+':8070/" target="_blank">'
url_text=url_text+'<img src="https://cdn.thenewstack.io/media/2017/10/60a587a1-nuclio.jpg" width="100" height="50"/>'
url_text=url_text+'<br>Click to open nuclio <br> in a new window</a></center></h1>'
ipywidgets.HTML(url_text)

---
# 3.0 Set Up Nuclio Pipelines with MONAI Models 
## 3.1 Start MONAI Label Server 
We need to have the MONAI Label server running before we deploy Nuclio pipelines.

In [ ]:
%cd $DLI_DATA_DIR
%env MONAI_LABEL_DATASTORE=cvat
%env MONAI_LABEL_DATASTORE_URL={"http://"+HOST_NAME+":8080"}
#### change user password below
%env MONAI_LABEL_DATASTORE_USERNAME=admin
%env MONAI_LABEL_DATASTORE_PASSWORD=admin
##Start monai Label server
!monailabel start_server \
  --app apps/endoscopy \
  --studies datasets/endoscopy_frames/ 

## 3.2 Set Up Nuclio Pipelines

In [ ]:
print(HOST_IP)

<div class="alert alert-danger"> 
You should have MONAI Label running above, then run commands below in a terminal.
</div> 

The command below runs the `nuctl` command, which would run `docker commands` to create multiple pipelines.
Each pipeline will be a new running container.

<div class="alert alert-block alert-info">
    
```
export MONAI_LABEL_SERVER=http://<place ip printed above here>:8000
echo $MONAI_LABEL_SERVER
```
</div>    

<div class="alert alert-danger"> 
Make sure `echo $MONAI_LABEL_SERVER` showed your IP address.
</div> 

Now run this command:

<div class="alert alert-block alert-info">
    
```
/usr/local/monailabel/plugins/cvat/deploy.sh sam2 interactor
docker ps |grep nuclio
```
</div>    

Now run this command: 

<div class="alert alert-block alert-info">    
    
```
/usr/local/monailabel/plugins/cvat/deploy.sh endoscopy tooltracking
docker ps |grep nuclio
```
</div>    

After model deployment, users can see the model names in the **Models** tag of CVAT.

<img src="../toolkit/tutorials/monailabel/figures/monailabel_endoscopy_cvat/cvat2.png" width="600"/>

To check or monitor the status of deployed function container, users can open Nuclio platform at http://HOST_NAME:8070 by default and see whether deployed models are running.

<img src="../toolkit/tutorials/monailabel/figures/monailabel_endoscopy_cvat/cvat3.png" width="600"/>

---
# 4.0 Active Learning Workflow with MONAI Label + CVAT

The advanced feature of the endoscopic application is to actively select and fine-tune models. 

Below is a step-by-step tutorial on using MONAI Label + CVAT for auto-segmentation, active learning, and fine-tuning models using the surgical tool segmentation bundle. a pretrained model is provided by the NGC bundle and loaded by MONAI Label server.


## 4.1 Start MONAI Label Server for Active Learning

Connect the MONAI Label server with CVAT datastore and URL, set USERNAME and PASSWORD to the same as the login credentials for CVAT.

In [ ]:
%cd $DLI_DATA_DIR
%env MONAI_LABEL_DATASTORE=cvat
%env MONAI_LABEL_DATASTORE_URL={"http://"+HOST_NAME+":8080"}
#### change user password below
%env MONAI_LABEL_DATASTORE_USERNAME=admin
%env MONAI_LABEL_DATASTORE_PASSWORD=admin

In [ ]:
# Specify the path of the downloaded endoscopy app (--app <APPPATH>) and the sample dataset (--studies <DATAPATH>)
!monailabel start_server \
  --app apps/endoscopy \
  --studies datasets/endoscopy_frames/ \
   --conf models tooltracking \
   --conf epistemic_enabled true \
   --conf epistemic_top_k 5

Then MONAI Label server will start scoring inference for each unlabeled image using the epistemic scoring method. Users can set the number of images to label in CVAT for each iteration.  The `--conf epistemic_top_k 5` option specifies the workload for each active learning loop.

MONAI Label server will automatically upload selected images to CVAT. Note: If images have been uploaded or the prior iteration's task is not complete, MONAI Label will prevent uploading more.

The automatically created project and tasks can be seen here: 

<img src="../toolkit/tutorials/monailabel/figures/monailabel_endoscopy_cvat/cvat8.png" width="600"/>

<img src="../toolkit/tutorials/monailabel/figures/monailabel_endoscopy_cvat/cvat9.png" width="600"/>


## 4.2 Labeling with CVAT and MONAI Label AI Tool

The created tasks will run inference automatically. Users can see the predicted masks in the job page, edit them, and follow the same annotating process as in Section 3. Once finished, click save, change job state, and finish the job.

<img src="../toolkit/tutorials/monailabel/figures/monailabel_endoscopy_cvat/cvat10.png" width="600"/>


## 4.3 Trigger Model Fine-Tuning

Continue to annotate until all created jobs are completed. MONAI Label will monitor the datastore status to check whether all jobs are in "completed" status.

Once all jobs are completed, the MONAI Label server will automatically trigger model fine-tuning and train the model with newly generated labels.

<img src="../toolkit/tutorials/monailabel/figures/monailabel_endoscopy_cvat/cvat11.png" width="600"/>


## 4.4 Scoring and Create Next Iteration for CVAT

Users will get a fine-tuned model if users provided the validation set.

<img src="../toolkit/tutorials/monailabel/figures/monailabel_endoscopy_cvat/cvat12.png" width="600"/>



## 4.5 Manually Upload Your Images and Create Task 
You can now use CVAT to create a task and upload your image. 

Hint: You can downlaod an image from this notebook from `/dli/data/datasets/endoscopy_frames` to your pc, 
then you can upload 1 or 2 to CVAT.

You should stop the MONAI Label server and see if you can trigger inference using Nuclio.


---
# 5.0 Shut Down CVAT
## 5.1 Remove Containers

In [ ]:
!docker ps -a | grep cvat

In [ ]:
## stop cvat and nuclio
!docker compose -p cvat down
!docker stop nuclio-local-storage-reader

In [ ]:
# removing pipelines 
# docker ps -a |grep nuclio |awk '{ print $1 }'
!docker ps -q --filter name=nuclio  | xargs docker stop
!docker ps -a -q --filter name=nuclio  | xargs docker rm

In [ ]:
### check containers are removed 
! docker ps -a |grep cvat
! docker ps -a |grep nuclio
## this cell should have no output

## 5.2 Remove Volumes

In [ ]:
##list docker volumes 
!docker volume ls|grep cvat
!docker volume ls|grep nuclio

In [ ]:
%%bash
docker volume rm $(docker volume ls --format '{{ .Name }}' | grep cvat)
docker volume rm $(docker volume ls --format '{{ .Name }}' | grep nuclio)

In [ ]:
## check no volumes remain. list docker volumes 
!docker volume ls|grep cvat
!docker volume ls|grep nuclio
## this cell should have no output

<img src="images/nvidia_header.png" style="margin-left: -30px; width: 300px; float: left;">